# SQL Market Data Analysis with DuckDB

This notebook applies SQL analysis to historical market data for AAPL, MSFT, and SPY using DuckDB.

The workflow covers:

- market data collection;
- normalization from wide to long format;
- data-quality validation;
- SQL-based return and performance analysis;
- aggregations, joins, CTEs, and window functions.

## Imports and configuration

In [1]:
import duckdb
import numpy as np
import pandas as pd
import yfinance as yf

## Data Collection

Historical adjusted OHLCV data are downloaded for AAPL, MSFT, and SPY
over the 2023–2024 period.

In [2]:
tickers = ["AAPL", "MSFT", "SPY"]

start_date = "2023-01-01"
end_date = "2025-01-01"
market_data_wide = yf.download(tickers, start_date, end_date, auto_adjust = True, progress = False, multi_level_index = True)

## Data normalization

The downloaded data uses a wide structure with a `Price–Ticker`
column MultiIndex. For SQL analysis, it is converted to long format
with one row per trading date and ticker.

In [3]:
# Move the ticker level from columns to rows
stacked_data = market_data_wide.stack()

# Convert Date and Ticker index levels into regular columns
market_data = stacked_data.reset_index()

# Normalize column names
market_data.columns = market_data.columns.str.lower()
market_data.columns.name = None

# Arrange columns according to the target analytical schema
market_data = market_data[
    ["date", "ticker", "open", "high", "low", "close", "volume"]]

## Data-quality validation

In [4]:
print(f"Shape: {market_data.shape}")
print(f"Duplicate date-ticker keys: {market_data[['date', 'ticker']].duplicated().sum()}")

display(market_data.head())
display(market_data.isna().sum().rename("missing_values"))
display(market_data["ticker"].value_counts().rename("observations"))

observations_per_ticker = market_data.groupby("ticker").size()

assert observations_per_ticker.nunique() == 1
assert observations_per_ticker.iloc[0] == 502

assert market_data[["date", "ticker"]].duplicated().sum() == 0
assert market_data.isna().sum().sum() == 0
assert market_data["ticker"].nunique() == 3

print("Data validation checks passed.")

Shape: (1506, 7)
Duplicate date-ticker keys: 0


,date,ticker,open,high,low,close,volume
0,2023-01-03,AAPL,127.995359,128.604481,121.992505,122.876724,112117500
1,2023-01-03,MSFT,235.907298,238.498511,230.394893,232.510574,25740000
2,2023-01-03,SPY,367.528413,369.498150,361.274962,364.133972,74850700
3,2023-01-04,AAPL,124.664809,126.403773,122.886552,124.144104,89113600
4,2023-01-04,MSFT,225.425972,225.998559,219.292468,222.339813,50623400


date      0
ticker    0
open      0
high      0
low       0
close     0
volume    0
Name: missing_values, dtype: int64

ticker
AAPL    502
MSFT    502
SPY     502
Name: observations, dtype: int64

Data validation checks passed.


## Dataset Overview

In [5]:
display(market_data.head(9))
market_data.info()

,date,ticker,open,high,low,close,volume
0,2023-01-03,AAPL,127.995359,128.604481,121.992505,122.876724,112117500
1,2023-01-03,MSFT,235.907298,238.498511,230.394893,232.510574,25740000
2,2023-01-03,SPY,367.528413,369.498150,361.274962,364.133972,74850700
3,2023-01-04,AAPL,124.664809,126.403773,122.886552,124.144104,89113600
4,2023-01-04,MSFT,225.425972,225.998559,219.292468,222.339813,50623400
5,2023-01-04,SPY,366.390526,368.972235,363.349869,366.945129,85934100
6,2023-01-05,AAPL,124.900613,125.529389,122.572179,122.827614,80962700
7,2023-01-05,MSFT,220.495844,220.835522,215.216364,215.750137,39585600
8,2023-01-05,SPY,364.994547,365.109284,362.164251,362.757080,76970500


<class 'pandas.DataFrame'>
RangeIndex: 1506 entries, 0 to 1505
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype        
---  ------  --------------  -----        
 0   date    1506 non-null   datetime64[s]
 1   ticker  1506 non-null   str          
 2   open    1506 non-null   float64      
 3   high    1506 non-null   float64      
 4   low     1506 non-null   float64      
 5   close   1506 non-null   float64      
 6   volume  1506 non-null   int64        
dtypes: datetime64[s](1), float64(4), int64(1), str(1)
memory usage: 82.5 KB


# SQL Analysis

Each section starts from a financial question, executes the corresponding
DuckDB query, validates the result, and provides a short interpretation.

## Analysis 1 - Daily Returns

**Financial question:** What were the daily simple returns of AAPL,
MSFT, and SPY during the 2023–2024 period?

**SQL concepts:** CTE, `LAG()`, `PARTITION BY`, temporal ordering.

Daily returns are calculated by comparing each closing price with the
previous available closing price for the same ticker.

In [6]:
daily_returns = duckdb.sql(""" 
WITH previous_prices AS (
SELECT date, ticker, close,
LAG(close) OVER (PARTITION BY ticker ORDER BY date) AS previous_close
FROM market_data
)
SELECT date, ticker, close, previous_close,
(close / previous_close) - 1 AS daily_return
FROM previous_prices
ORDER BY ticker, date
""").df()

display(daily_returns)

,date,ticker,close,previous_close,daily_return
0,2023-01-03,AAPL,122.876724,NaN,NaN
1,2023-01-04,AAPL,124.144104,122.876724,0.010314
2,2023-01-05,AAPL,122.827614,124.144104,-0.010605
3,2023-01-06,AAPL,127.346931,122.827614,0.036794
4,2023-01-09,AAPL,127.867645,127.346931,0.004089
...,...,...,...,...,...
1501,2024-12-24,SPY,591.179077,584.680359,0.011115
1502,2024-12-26,SPY,591.218445,591.179077,0.000067
1503,2024-12-27,SPY,584.994995,591.218445,-0.010526
1504,2024-12-30,SPY,578.319275,584.994995,-0.011412


In [7]:
print(daily_returns.shape)
display(daily_returns.head())
display(daily_returns.groupby("ticker").head(3))
print(daily_returns.isna().sum())

assert daily_returns.shape[0] == market_data.shape[0]
assert daily_returns["ticker"].nunique() == 3

previous_close_nulls = (
    daily_returns["previous_close"]
    .isna()
    .groupby(daily_returns["ticker"])
    .sum()
)

daily_return_nulls = (
    daily_returns["daily_return"]
    .isna()
    .groupby(daily_returns["ticker"])
    .sum()
)

assert previous_close_nulls.eq(1).all()
assert daily_return_nulls.eq(1).all()

dates_are_ordered = (
    daily_returns
    .groupby("ticker")["date"]
    .apply(lambda dates: dates.is_monotonic_increasing)
)

assert dates_are_ordered.all()

print("Daily-return validation checks passed.")

(1506, 5)


,date,ticker,close,previous_close,daily_return
0,2023-01-03,AAPL,122.876724,NaN,NaN
1,2023-01-04,AAPL,124.144104,122.876724,0.010314
2,2023-01-05,AAPL,122.827614,124.144104,-0.010605
3,2023-01-06,AAPL,127.346931,122.827614,0.036794
4,2023-01-09,AAPL,127.867645,127.346931,0.004089


,date,ticker,close,previous_close,daily_return
0,2023-01-03,AAPL,122.876724,NaN,NaN
1,2023-01-04,AAPL,124.144104,122.876724,0.010314
2,2023-01-05,AAPL,122.827614,124.144104,-0.010605
502,2023-01-03,MSFT,232.510574,NaN,NaN
503,2023-01-04,MSFT,222.339813,232.510574,-0.043743
504,2023-01-05,MSFT,215.750137,222.339813,-0.029638
1004,2023-01-03,SPY,364.133972,NaN,NaN
1005,2023-01-04,SPY,366.945129,364.133972,0.007720
1006,2023-01-05,SPY,362.757080,366.945129,-0.011413


date              0
ticker            0
close             0
previous_close    3
daily_return      3
dtype: int64
Daily-return validation checks passed.


### Manual control of operation

In [8]:
aapl_first_rows = (
    daily_returns[daily_returns["ticker"] == "AAPL"]
    .head(2)
)

current_close = aapl_first_rows.iloc[1]["close"]
previous_close = aapl_first_rows.iloc[1]["previous_close"]
sql_return = aapl_first_rows.iloc[1]["daily_return"]

expected_return = current_close / previous_close - 1

print("Expected return:", expected_return)
print("SQL return:", sql_return)

assert np.isclose(expected_return, sql_return)

Expected return: 0.010314237855447272
SQL return: 0.010314237855447272


The query preserves the original date–ticker granularity and adds the
previous closing price and daily simple return to each observation.

The first observation for each ticker has a structurally missing return
because no earlier closing price is available. All subsequent returns are
calculated independently within each ticker's chronological sequence.

## Analysis 2 — Monthly Return Summary

**Financial question:** How did average daily return and daily volatility
vary by month for each ticker?

**SQL concepts:** `DATE_TRUNC`, `GROUP BY`, aggregate functions, null handling.

Daily returns are aggregated by ticker and calendar month. The resulting
metrics describe the distribution of daily returns within each month and
should not be interpreted as compounded monthly performance.

In [9]:
monthly_summary = duckdb.sql(""" 
SELECT DATE_TRUNC('month', date) AS month, 
ticker,
COUNT(daily_return) AS trading_days,
AVG(daily_return) AS avg_daily_return,
STDDEV_SAMP(daily_return) AS daily_volatility,
MIN(daily_return) AS min_daily_return,
MAX(daily_return) AS max_daily_return
FROM daily_returns
WHERE daily_return IS NOT NULL
GROUP BY month, ticker
ORDER BY ticker, month
""").df()

display(monthly_summary)

,month,ticker,trading_days,avg_daily_return,daily_volatility,min_daily_return,max_daily_return
0,2023-01-01,AAPL,19,0.007633,0.013107,-0.020078,0.036794
1,2023-02-01,AAPL,19,0.001336,0.016534,-0.026680,0.037063
2,2023-03-01,AAPL,23,0.004977,0.013760,-0.014915,0.035090
3,2023-04-01,AAPL,19,0.001580,0.012627,-0.015973,0.034104
4,2023-05-01,AAPL,22,0.002124,0.012628,-0.015155,0.046927
...,...,...,...,...,...,...,...
67,2024-08-01,SPY,22,0.001120,0.012039,-0.029124,0.023117
68,2024-09-01,SPY,20,0.001075,0.008584,-0.020579,0.017064
69,2024-10-01,SPY,23,-0.000366,0.007004,-0.019603,0.009458
70,2024-11-01,SPY,20,0.002927,0.007489,-0.012809,0.024865


In [10]:
print(monthly_summary.shape)
display(monthly_summary.head(12))
print(monthly_summary.isna().sum())

assert monthly_summary.shape[0] == 72
assert monthly_summary["ticker"].nunique() == 3
assert monthly_summary["month"].nunique() == 24
assert monthly_summary[["month", "ticker"]].duplicated().sum() == 0

months_per_ticker = (
    monthly_summary
    .groupby("ticker")["month"]
    .nunique()
)

display(months_per_ticker)
assert months_per_ticker.eq(24).all()

(72, 7)


,month,ticker,trading_days,avg_daily_return,daily_volatility,min_daily_return,max_daily_return
0,2023-01-01,AAPL,19,0.007633,0.013107,-0.020078,0.036794
1,2023-02-01,AAPL,19,0.001336,0.016534,-0.026680,0.037063
2,2023-03-01,AAPL,23,0.004977,0.013760,-0.014915,0.035090
3,2023-04-01,AAPL,19,0.001580,0.012627,-0.015973,0.034104
4,2023-05-01,AAPL,22,0.002124,0.012628,-0.015155,0.046927
5,2023-06-01,AAPL,21,0.004345,0.009595,-0.007756,0.023103
6,2023-07-01,AAPL,20,0.000664,0.007735,-0.010856,0.017306
7,2023-08-01,AAPL,23,-0.001754,0.016215,-0.048020,0.021949
8,2023-09-01,AAPL,20,-0.004534,0.014329,-0.035794,0.016913
9,2023-10-01,AAPL,22,-0.000066,0.010352,-0.024605,0.014835


month               0
ticker              0
trading_days        0
avg_daily_return    0
daily_volatility    0
min_daily_return    0
max_daily_return    0
dtype: int64


ticker
AAPL    24
MSFT    24
SPY     24
Name: month, dtype: int64

## Analysis 3 — Sector-Level Risk and Return Summary

**Financial question:** How did average daily return and daily volatility vary across market sectors (Technology vs. Benchmark Index)?

**SQL concepts:** `JOIN`, inline mapping with `VALUES` / reference tables, aggregate functions, `GROUP BY`.

Individual asset daily returns are enriched with sectoral metadata via an inner join. 
Aggregating at the sector level provides equal-weighted portfolio characteristics across asset classes.

In [11]:
sector_df = pd.DataFrame({
    "ticker" : ["AAPL", "MSFT", "SPY"],
    "sector" : ["Technology", "Technology", "ETF"]
})

display(sector_df)

,ticker,sector
0,AAPL,Technology
1,MSFT,Technology
2,SPY,ETF


In [12]:
sector_summary = duckdb.sql("""
SELECT sectors.sector,
COUNT(day_return.daily_return) AS total_observations,
AVG(day_return.daily_return) AS avg_daily_return,
STDDEV_SAMP(day_return.daily_return) AS daily_volatility,
MIN(day_return.daily_return) AS min_daily_return,
MAX(day_return.daily_return) AS max_daily_return
FROM daily_returns day_return
JOIN sector_df sectors
ON day_return.ticker = sectors.ticker
WHERE daily_return IS NOT NULL
GROUP BY sectors.sector""").df()

display(sector_summary)

,sector,total_observations,avg_daily_return,daily_volatility,min_daily_return,max_daily_return
0,Technology,1002,0.001380,0.013869,-0.060528,0.072649
1,ETF,501,0.000949,0.008075,-0.029804,0.024865


In [13]:
# Structured Controls
assert sector_summary.shape[0] == 2
assert set(sector_summary["sector"]) == {"Technology", "ETF"}
assert sector_summary["total_observations"].sum() == 1503  # 1506 totali - 3 NULL iniziali
assert sector_summary.isna().sum().sum() == 0

## Analysis 4 — 30-Day Rolling Moving Average

**Financial question:** How did the 30-trading-day simple moving average (SMA) of close prices evolve over time for each ticker?

**SQL concepts:** Window functions, `OVER()`, `PARTITION BY`, `ORDER BY`, explicit window frames (`ROWS BETWEEN 29 PRECEDING AND CURRENT ROW`).

A 30-trading-day rolling average filters high-frequency market noise to reveal medium-term price trends. 
The window function preserves the native daily granularity while appending dynamic statistical features.

In [14]:
price_moving_averages = duckdb.sql("""
SELECT date, ticker, close,
AVG(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS sma_30
FROM market_data
ORDER BY ticker, date
""").df()

display(price_moving_averages)

,date,ticker,close,sma_30
0,2023-01-03,AAPL,122.876724,122.876724
1,2023-01-04,AAPL,124.144104,123.510414
2,2023-01-05,AAPL,122.827614,123.282814
3,2023-01-06,AAPL,127.346931,124.298843
4,2023-01-09,AAPL,127.867645,125.012604
...,...,...,...,...
1501,2024-12-24,SPY,591.179077,586.636802
1502,2024-12-26,SPY,591.218445,586.847927
1503,2024-12-27,SPY,584.994995,586.842133
1504,2024-12-30,SPY,578.319275,586.739237


In [15]:
print(price_moving_averages.shape)
display(price_moving_averages.head(10))
print(price_moving_averages.isna().sum())

# 1. Structural Integrity
assert price_moving_averages.shape == (1506, 4)
assert price_moving_averages["ticker"].nunique() == 3
assert price_moving_averages["sma_30"].isna().sum() == 0
assert price_moving_averages[["date", "ticker"]].duplicated().sum() == 0

# 2. Quantitative consistency check on Cold Start (row 0 for every ticker: close == sma_30)
first_rows = price_moving_averages.groupby("ticker").first()
assert np.isclose(first_rows["close"], first_rows["sma_30"]).all()

# 3. Cross-validation using pandas' `rolling()` function on a single asset
aapl_sql = price_moving_averages[price_moving_averages["ticker"] == "AAPL"]["sma_30"].reset_index(drop=True)
aapl_pandas = (
    market_data[market_data["ticker"] == "AAPL"]
    .sort_values("date")["close"]
    .rolling(window=30, min_periods=1)
    .mean()
    .reset_index(drop=True)
)
assert np.isclose(aapl_sql, aapl_pandas).all()

(1506, 4)


,date,ticker,close,sma_30
0,2023-01-03,AAPL,122.876724,122.876724
1,2023-01-04,AAPL,124.144104,123.510414
2,2023-01-05,AAPL,122.827614,123.282814
3,2023-01-06,AAPL,127.346931,124.298843
4,2023-01-09,AAPL,127.867645,125.012604
5,2023-01-10,AAPL,128.437469,125.583415
6,2023-01-11,AAPL,131.149109,126.378514
7,2023-01-12,AAPL,131.070496,126.965012
8,2023-01-13,AAPL,132.396790,127.568542
9,2023-01-17,AAPL,133.556107,128.167299


date      0
ticker    0
close     0
sma_30    0
dtype: int64


## Analysis 5 — Asset Cumulative Performance Ranking

**Financial question:** What was the total cumulative return of each asset over the 2023–2024 period, and how do they rank against each other?

**SQL concepts:** `FIRST_VALUE()`, `LAST_VALUE()`, window frames (`ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`), CTEs, and ranking functions (`RANK()`, `DENSE_RANK()`, `ROW_NUMBER()`).

Cumulative return measures holding period performance from the initial trading day to the final observation. 
Ranking window functions provide an ordinal hierarchy of relative alpha generation across assets.

In [16]:
asset_performance_rank = duckdb.sql(""" 
WITH price_boundaries AS (
SELECT DISTINCT ticker, 
MIN(date) OVER (PARTITION BY ticker) AS start_date,
MAX(date) OVER (PARTITION BY ticker) AS end_date,
FIRST_VALUE(close) OVER (PARTITION BY ticker ORDER BY date) AS start_price,
LAST_VALUE(close) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS end_price
FROM market_data
)
SELECT ticker, start_date, end_date, start_price, end_price,
(end_price / start_price) - 1 AS cumulative_return,
RANK() OVER (ORDER BY (end_price / start_price) - 1 DESC) AS performance_rank
FROM price_boundaries
ORDER BY performance_rank
""").df()

display(asset_performance_rank)


,ticker,start_date,end_date,start_price,end_price,cumulative_return,performance_rank
0,AAPL,2023-01-03,2024-12-31,122.876724,248.615784,1.023294,1
1,MSFT,2023-01-03,2024-12-31,232.510574,415.775665,0.788201,2
2,SPY,2023-01-03,2024-12-31,364.133972,576.215271,0.582427,3


In [17]:
print(asset_performance_rank.shape)
display(asset_performance_rank)
print(asset_performance_rank.isna().sum())

# 1. Structural integrity checks
assert asset_performance_rank.shape == (3, 7)
assert asset_performance_rank["ticker"].nunique() == 3
assert asset_performance_rank.isna().sum().sum() == 0

# 2. Ranking consistency checks
assert list(asset_performance_rank["performance_rank"]) == [1, 2, 3]
assert asset_performance_rank["cumulative_return"].is_monotonic_decreasing

# 3. Quantitative cross-validation against raw market data for AAPL
aapl_raw = market_data[market_data["ticker"] == "AAPL"].sort_values("date")
expected_aapl_return = (aapl_raw["close"].iloc[-1] / aapl_raw["close"].iloc[0]) - 1

sql_aapl_return = asset_performance_rank.loc[
    asset_performance_rank["ticker"] == "AAPL", "cumulative_return"
].values[0]

assert np.isclose(sql_aapl_return, expected_aapl_return)

(3, 7)


,ticker,start_date,end_date,start_price,end_price,cumulative_return,performance_rank
0,AAPL,2023-01-03,2024-12-31,122.876724,248.615784,1.023294,1
1,MSFT,2023-01-03,2024-12-31,232.510574,415.775665,0.788201,2
2,SPY,2023-01-03,2024-12-31,364.133972,576.215271,0.582427,3


ticker               0
start_date           0
end_date             0
start_price          0
end_price            0
cumulative_return    0
performance_rank     0
dtype: int64


## Analysis 6 — Portfolio Benchmark Comparison & Outperformance

**Financial question:** Which assets achieved an average daily return strictly above the equal-weighted portfolio benchmark, and what was their excess return (spread)?

**SQL concepts:** Subqueries, Common Table Expressions (CTEs), scalar aggregations, filtering with derived thresholds.

A portfolio-wide average daily return is derived as an equal-weighted benchmark. 
Assets are evaluated against this threshold to identify alpha generators relative to the collective investment universe.

In [18]:
portfolio_outperformers = duckdb.sql(""" 
WITH daily_mean AS (
SELECT ticker,
AVG(daily_return) AS avg_daily_return
FROM daily_returns
WHERE daily_return IS NOT NULL
GROUP BY ticker
),
portfolio_mean AS (
SELECT
AVG(daily_return) AS portfolio_benchmark_avg
FROM daily_returns
WHERE daily_return IS NOT NULL
)
SELECT ticker, avg_daily_return,
avg_daily_return - portfolio_benchmark_avg AS excess_daily_return,
portfolio_benchmark_avg
FROM daily_mean
CROSS JOIN portfolio_mean
WHERE avg_daily_return > portfolio_benchmark_avg
""").df()

display(portfolio_outperformers)

,ticker,avg_daily_return,excess_daily_return,portfolio_benchmark_avg
0,AAPL,0.001498,0.000261,0.001236
1,MSFT,0.001263,0.000026,0.001236


In [19]:
print(portfolio_outperformers.shape)
display(portfolio_outperformers)
print(portfolio_outperformers.isna().sum())

# 1. Structural integrity checks
assert portfolio_outperformers.shape[0] in [1, 2]  # Subset of the 3 tickers
assert portfolio_outperformers.isna().sum().sum() == 0

# 2. Outperformance logic validation
assert (portfolio_outperformers["excess_daily_return"] > 0).all()
assert (portfolio_outperformers["avg_daily_return"] > portfolio_outperformers["portfolio_benchmark_avg"]).all()

# 3. Cross-validation against pure Pandas benchmark calculation
valid_returns = daily_returns.dropna(subset=["daily_return"])
pandas_global_avg = valid_returns["daily_return"].mean()

sql_benchmark_val = portfolio_outperformers["portfolio_benchmark_avg"].iloc[0]
assert np.isclose(sql_benchmark_val, pandas_global_avg)

(2, 4)


,ticker,avg_daily_return,excess_daily_return,portfolio_benchmark_avg
0,AAPL,0.001498,0.000261,0.001236
1,MSFT,0.001263,0.000026,0.001236


ticker                     0
avg_daily_return           0
excess_daily_return        0
portfolio_benchmark_avg    0
dtype: int64


## Executive Summary & Analytical Key Takeaways

1. **Cumulative Performance Hierarchy (2023–2024):**
   - **AAPL** led total performance with a cumulative return of **+102.33%** ($122.88 \to 248.62$).
   - **MSFT** delivered **+78.82%** ($232.51 \to 415.78$), driven by continuous medium-term momentum.
   - **SPY (Benchmark)** recorded **+58.24%** ($364.13 \to 576.22$), illustrating the broad equity market rally.

2. **Sector-Level Risk & Return Trade-off:**
   - The **Technology sector** achieved an average daily return of **+0.138%** with a daily volatility of **1.39%**.
   - The broad-market **ETF sector** exhibited a lower average daily return of **+0.095%**, but with substantially lower risk (**0.81%** daily volatility), confirming classical modern portfolio theory dynamics.

3. **Active Outperformance (Alpha Generation):**
   - Relative to the equal-weighted portfolio benchmark (+0.1236% daily mean), both **AAPL (+0.0261% daily alpha)** and **MSFT (+0.0026% daily alpha)** generated positive active spread over the 2-year window.

4. **Technical & Engineering Architecture:**
   - Zero ETL friction achieved via DuckDB querying directly over in-memory pandas DataFrames.
   - Robust analytical pipeline leveraging SQL window frames (`ROWS BETWEEN`), relational joins, and multi-step CTE benchmarks validated against analytical assertions.